# Plotting parallel trends
Este script lo estoy haciendo inmediatamente desupués de haber hecho el de `parallel-trends`, pero ahora lo que voy a hacer es graficar las tendencias antes y después de la intervención, tanto para controles como para el resto de unidades.

In [1]:
import os
os.chdir("/Users/mariano/Documents/itam/tesis/speed-cameras/scripts/")

In [2]:
from ps_features_builder import PSFeaturesBuilder
from ps_matching import PSMatching
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import pickle

In [3]:
PATH_DATA = "../data/"
PATHS = {
    'vialidades': os.path.join(PATH_DATA, 'vialidades.json'),
    'speed_cameras': os.path.join(PATH_DATA, 'fotocivicas-ubicacion-puntos', 'fotocivicas-ubicacion-puntos.shp'),
    'metro_coordinates': os.path.join(PATH_DATA, 'metro-station-coordinates.parquet'),
    'afluencia_metro': os.path.join(PATH_DATA, 'afluencia-metro-semanal.parquet'),
    'classified_incidents': os.path.join(PATH_DATA, 'classified-incidents.parquet'),
    'volumen_mensual': os.path.join(PATH_DATA, 'volumen-total-mensual.parquet'),
    'ps_feature_builders' : os.path.join(PATH_DATA, "pickle_objects", "ps_features_builder")
}

# grid_type, radio, n_circles
radios = [
    ("circular", 37.5, 25), # 0
    ("circular", 50, 15), # 1
    ("circular", 100, 20), # 2
    ("circular", 150, 15), # 3
    ("circular", 200, 20), # 4
    ("circular", 250, 25), # 5
]

object_name_format = "{grid_type}_{radio}_{n_circles}.pkl"

### Muestra original
#### Sin propensity score matching
Aquí voy a hacer la gráfica de cómo se ven las tendencias de accidentes mensuales para todas las unidades tratadas vs todas las unidades de control.  
Voy a graficar el promedio mensual de accidentes.

In [4]:
grid_type, radio, n_circles = radios[0]
filename = object_name_format.format(
    grid_type=grid_type,
    radio=int(radio),
    n_circles=n_circles
)
obj_path = os.path.join(PATHS.get("ps_feature_builders"), filename)
with open(obj_path, "rb") as f:
    ps_builder : PSFeaturesBuilder = pickle.load(f)

psm = PSMatching(
    ps_features=ps_builder.ps_features,
    grid=ps_builder.grid,
    outcome=ps_builder.outcome,
    grid_type=ps_builder.grid_type,
    grid_size=ps_builder.circle_radius
)
psm.build(matching_method="nearest", n_matches=1)

In [5]:
def build_pdf(
    df_camera_assignment : pd.DataFrame
) -> pd.DataFrame:
    """
    Args:
        - df_camera_assignment
            pandas dataframe with columns {grid_id, has_camera}
            has_camera must have values either 0 or 1
    Return: 
        - Dataframe with columns {'unit_type', 'timestamp', 'total', 'min', 'pic', 'fcs'}
    """
    pdf = (
        ps_builder
        .outcome
        .pipe(lambda df: df[
            (df.timestamp >= ps_builder.inicio_operaciones - pd.Timedelta(days=365*3)) &
            (df.timestamp <= ps_builder.inicio_operaciones + pd.Timedelta(days=365*3))
        ])
        .merge(
            df_camera_assignment,
            on="grid_id",
            how="inner"
        )
        .assign(unit_type=lambda x: x.has_camera.map({0:"Control", 1:"Tratamiento"}))
        .groupby(["unit_type", "timestamp"])
        .agg(
            total=pd.NamedAgg("total", "mean"),
            min=pd.NamedAgg("min", "mean"),
            pic=pd.NamedAgg("pic", "mean"),
            fcs=pd.NamedAgg("fcs", "mean")
        )
        .reset_index()
    )
    return pdf

In [7]:
for grid_type, radio, n_circles in radios:
    filename = object_name_format.format(
        grid_type=grid_type,
        radio=int(radio),
        n_circles=n_circles
    )
    obj_path = os.path.join(PATHS.get("ps_feature_builders"), filename)
    with open(obj_path, "rb") as f:
        ps_builder : PSFeaturesBuilder = pickle.load(f)
    
    psm = PSMatching(
        ps_features=ps_builder.ps_features,
        grid=ps_builder.grid,
        outcome=ps_builder.outcome,
        grid_type=ps_builder.grid_type,
        grid_size=ps_builder.circle_radius
    )
    psm.build(matching_method="nearest", n_matches=1)
    
    pdf_no_psm = build_pdf(
        df_camera_assignment=ps_builder.ps_features[["grid_id", "has_camera"]]
    )
    
    pdf_with_psm = build_pdf(
        df_camera_assignment=psm.matched_grids[["grid_id", "has_camera"]]
    )

    # Y ahora solo es tema de hacer una gráfica bonita que tenga 4 filas y 2 columnas
    # En la primera columna voy a poner la gráfica de todas las unidades
    # y en la segunda voy a poner las que ya están matcheadas
    
    fig, axes = plt.subplots(4,2, figsize=(10, 12), sharey=False, sharex=True) 
    
    for row, accident_type in enumerate(["total", "min", "pic", "fcs"]):
        for col, (title, df) in enumerate([("SIN PSM", pdf_no_psm), ("CON PSM", pdf_with_psm)]):
            ax = axes[row][col]
            #ax.spines.top.set_visible(False)
            #ax.spines.right.set_visible(False)
            ax.grid(axis='y', alpha=.2, linestyle=':')
    
            unit_types = [("Control", "#a0aab2"), ("Tratamiento", "black")]
            for (ut, color) in unit_types:
                ax.plot(
                    df[df.unit_type == ut].timestamp,
                    df[df.unit_type == ut][accident_type],
                    color=color,
                    label=ut,
                    linewidth=.8
                )
            ax.legend(ncols=2, frameon=False, fontsize="small")
            ax.vlines(
                x=pd.Timestamp("2019-04-22"), 
                ymin=df[accident_type].min(), 
                ymax=df[accident_type].max(),
                zorder=-10, color="gray",
                linestyles=":",
                linewidth=1, alpha=.8
            )
            ax.set_title(f"{accident_type.upper()} {title}", size=10, loc="left")
    
    fig.supylabel("Promedio mensual de accidentes")
    fig.suptitle(
        "Comparativo por tipo de accidentes con y sin PSM", 
        y=.99, ha="left", x=0.088
    )
    image_name = "trend_mean_accidents_{radio}.png".format(radio=int(radio))
    fig.tight_layout()
    fig.savefig(
        os.path.join(PATH_DATA, "graphs/mean_accidents_trends", image_name), 
        dpi=300, transparent=True
    )
    plt.close()